In [1]:
from pathlib import Path
import json
import re
import time
import torch

from transformers import AutoProcessor, AutoModelForImageTextToText


# ============================================================
# SETTINGS
# ============================================================

MODEL_ID = "google/medgemma-1.5-4b-it"

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

consultation_id = "day1_consultation01"

transcript_path = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
    / f"{consultation_id}_transcript.txt"
)

output_dir = (
    project
    / "results/nlp/medgemma_1_5_4b_it/compatibility_test"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# VERIFY INPUT
# ============================================================

print("=" * 75)
print("MEDGEMMA 1.5 CLINICAL EXTRACTION COMPATIBILITY TEST")
print("=" * 75)

print("Transcript exists:", transcript_path.exists())

if not transcript_path.exists():
    raise FileNotFoundError(transcript_path)

transcript = transcript_path.read_text(
    encoding="utf-8"
)

print("Transcript characters:", len(transcript))


# ============================================================
# LOAD MODEL
# ============================================================

print("\nLoading MedGemma...")

load_start = time.time()

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

load_seconds = time.time() - load_start

print("Model loaded successfully")
print("Load time:", round(load_seconds, 2), "seconds")
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)


# ============================================================
# EXTRACTION PROMPT
# ============================================================

prompt = f"""
You are extracting structured clinical information from a
doctor-patient consultation transcript.

IMPORTANT RULES:

1. Use ONLY information explicitly stated in the transcript.
2. Do NOT invent, infer, assume, or complete missing information.
3. Preserve negations exactly.
4. Preserve numbers, durations, doses and frequencies exactly
   when they are stated.
5. Every extracted clinical item MUST include a short
   evidence_quote copied from the transcript.
6. If information is not stated, use an empty list.
7. Return ONLY valid JSON.
8. Do not use Markdown or code fences.

Return exactly this JSON structure:

{{
  "consultation_id": "{consultation_id}",

  "presenting_complaints": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ],

  "symptoms": [
    {{
      "fact": "",
      "status": "present_or_absent",
      "duration": "",
      "severity": "",
      "evidence_quote": ""
    }}
  ],

  "medications": [
    {{
      "name": "",
      "dose": "",
      "frequency": "",
      "status": "",
      "evidence_quote": ""
    }}
  ],

  "allergies": [
    {{
      "substance": "",
      "reaction": "",
      "status": "",
      "evidence_quote": ""
    }}
  ],

  "medical_history": [
    {{
      "fact": "",
      "status": "",
      "evidence_quote": ""
    }}
  ],

  "family_history": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ],

  "social_history": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ],

  "assessment": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ],

  "plan": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ],

  "safety_netting": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ]
}}

CONSULTATION TRANSCRIPT:

{transcript}
"""


messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]


# ============================================================
# TOKENIZE
# ============================================================

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

input_tokens = inputs["input_ids"].shape[-1]

print("\nInput tokens:", input_tokens)


# ============================================================
# GENERATE
# ============================================================

print("\nGenerating structured clinical extraction...")

torch.cuda.reset_peak_memory_stats()

generation_start = time.time()

with torch.inference_mode():

    output = model.generate(
        **inputs,
        max_new_tokens=1800,
        do_sample=False
    )

generation_seconds = (
    time.time()
    - generation_start
)

generated_tokens = (
    output.shape[-1]
    - input_tokens
)

response = processor.decode(
    output[0][input_tokens:],
    skip_special_tokens=True
).strip()

peak_gpu_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)


# ============================================================
# CLEAN + PARSE JSON
# ============================================================

cleaned = response.strip()

cleaned = re.sub(
    r"^```(?:json)?\s*",
    "",
    cleaned,
    flags=re.IGNORECASE
)

cleaned = re.sub(
    r"\s*```$",
    "",
    cleaned
)

json_valid = False
parsed = None
json_error = None

try:
    parsed = json.loads(cleaned)
    json_valid = True

except Exception as e:
    json_error = str(e)


# ============================================================
# SAVE RESULTS
# ============================================================

raw_path = (
    output_dir
    / f"{consultation_id}_medgemma_raw.txt"
)

raw_path.write_text(
    response,
    encoding="utf-8"
)

if json_valid:

    json_path = (
        output_dir
        / f"{consultation_id}_medgemma_extraction.json"
    )

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            parsed,
            f,
            indent=2,
            ensure_ascii=False
        )

metadata = {
    "consultation_id": consultation_id,
    "model": MODEL_ID,
    "input_tokens": input_tokens,
    "generated_tokens": generated_tokens,
    "generation_seconds": generation_seconds,
    "peak_gpu_gb": peak_gpu_gb,
    "json_valid": json_valid,
    "json_error": json_error
}

metadata_path = (
    output_dir
    / f"{consultation_id}_medgemma_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 75)
print("MEDGEMMA COMPATIBILITY TEST RESULT")
print("=" * 75)

print("Consultation:", consultation_id)
print("JSON valid:", json_valid)
print("Generated tokens:", generated_tokens)
print(
    "Generation time:",
    round(generation_seconds, 2),
    "seconds"
)
print(
    "Peak GPU memory:",
    round(peak_gpu_gb, 2),
    "GB"
)

if json_error:
    print("JSON error:", json_error)

print("\nSTRUCTURED OUTPUT")
print("-" * 75)
print(cleaned)

print("\nSaved:")
print(raw_path)

if json_valid:
    print(json_path)

print(metadata_path)


MEDGEMMA 1.5 CLINICAL EXTRACTION COMPATIBILITY TEST
Transcript exists: True
Transcript characters: 6975

Loading MedGemma...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Model loaded successfully
Load time: 77.76 seconds
GPU: NVIDIA H200 NVL

Input tokens: 2264

Generating structured clinical extraction...

MEDGEMMA COMPATIBILITY TEST RESULT
Consultation: day1_consultation01
JSON valid: False
Generated tokens: 1800
Generation time: 54.72 seconds
Peak GPU memory: 8.52 GB
JSON error: Expecting value: line 1 column 1 (char 0)

STRUCTURED OUTPUT
---------------------------------------------------------------------------
<unused94>thought
The user wants me to extract clinical information from the provided doctor-patient consultation transcript and structure it into a JSON object according to the specified rules.

**Plan:**

1.  **Identify the main sections:** Presenting complaints, symptoms, medications, allergies, medical history, family history, social history, assessment, plan, and safety netting.
2.  **Extract information for each section:**
    *   **Presenting complaints:** The patient's main reason for seeking help.
    *   **Symptoms:** Specific sig

In [2]:
import json
import re
import time
import torch
from pathlib import Path


# ============================================================
# MEDGEMMA COMPATIBILITY TEST — RETRY
# ============================================================

print("=" * 75)
print("MEDGEMMA CLINICAL EXTRACTION RETRY")
print("=" * 75)

# Make sure previous model is still loaded
if "model" not in globals() or "processor" not in globals():
    raise RuntimeError(
        "MedGemma is not loaded. Re-run the previous model-loading cell first."
    )

consultation_id = "day1_consultation01"

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

transcript_path = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
    / f"{consultation_id}_transcript.txt"
)

output_dir = (
    project
    / "results/nlp/medgemma_1_5_4b_it/compatibility_test"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

transcript = transcript_path.read_text(
    encoding="utf-8"
)


# ============================================================
# IMPROVED COMPACT PROMPT
# ============================================================

prompt = f"""
Extract clinically important information from this doctor-patient
consultation transcript.

STRICT RULES:

1. Use only information explicitly stated in the transcript.
2. Do not infer or invent information.
3. Preserve important negations.
4. Preserve doses, numbers, durations and frequencies.
5. Every item must contain a short evidence_quote from the transcript.
6. Combine closely related information instead of creating duplicates.
7. For status, use only "present" or "absent".
8. If a category has no information, return [].
9. Return valid JSON only.
10. Do not include explanation, analysis, planning or Markdown.

Required JSON structure:

{{
  "consultation_id": "{consultation_id}",

  "presenting_complaints": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ],

  "symptoms": [
    {{
      "fact": "",
      "status": "present",
      "details": "",
      "evidence_quote": ""
    }}
  ],

  "medications": [
    {{
      "fact": "",
      "status": "present",
      "evidence_quote": ""
    }}
  ],

  "allergies": [
    {{
      "fact": "",
      "status": "present",
      "evidence_quote": ""
    }}
  ],

  "medical_history": [
    {{
      "fact": "",
      "status": "present",
      "evidence_quote": ""
    }}
  ],

  "family_history": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ],

  "social_history": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ],

  "assessment": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ],

  "plan": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ],

  "safety_netting": [
    {{
      "fact": "",
      "evidence_quote": ""
    }}
  ]
}}

TRANSCRIPT:

{transcript}
"""

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]


# ============================================================
# TOKENIZE
# ============================================================

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

input_tokens = inputs["input_ids"].shape[-1]

print("Input tokens:", input_tokens)


# ============================================================
# GENERATE
# ============================================================

torch.cuda.reset_peak_memory_stats()

start = time.time()

with torch.inference_mode():

    output = model.generate(
        **inputs,
        max_new_tokens=3200,
        do_sample=False
    )

generation_seconds = time.time() - start

generated_tokens = (
    output.shape[-1]
    - input_tokens
)

full_response = processor.decode(
    output[0][input_tokens:],
    skip_special_tokens=True
).strip()

peak_gpu_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)


# ============================================================
# EXTRACT ONLY FINAL JSON
# ============================================================

# MedGemma may place its final response after <unused95>.
# Anything before that is discarded.
if "<unused95>" in full_response:
    final_response = full_response.rsplit(
        "<unused95>",
        1
    )[-1].strip()
else:
    final_response = full_response.strip()

# Remove Markdown fences if produced
final_response = re.sub(
    r"^```(?:json)?\s*",
    "",
    final_response,
    flags=re.IGNORECASE
)

final_response = re.sub(
    r"\s*```$",
    "",
    final_response
).strip()

# Keep only JSON object
first_brace = final_response.find("{")
last_brace = final_response.rfind("}")

if first_brace != -1 and last_brace != -1:
    json_text = final_response[
        first_brace:last_brace + 1
    ]
else:
    json_text = final_response


# ============================================================
# PARSE JSON
# ============================================================

json_valid = False
parsed = None
json_error = None

try:
    parsed = json.loads(json_text)
    json_valid = True
except Exception as e:
    json_error = str(e)


# ============================================================
# EVIDENCE-GROUNDING CHECK
# ============================================================

def normalize_match(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


evidence_total = 0
evidence_matched = 0

if json_valid:

    normalized_transcript = normalize_match(
        transcript
    )

    categories = [
        "presenting_complaints",
        "symptoms",
        "medications",
        "allergies",
        "medical_history",
        "family_history",
        "social_history",
        "assessment",
        "plan",
        "safety_netting"
    ]

    for category in categories:

        for item in parsed.get(
            category,
            []
        ):

            quote = item.get(
                "evidence_quote",
                ""
            )

            if quote:

                evidence_total += 1

                normalized_quote = normalize_match(
                    quote
                )

                if (
                    normalized_quote
                    and normalized_quote
                    in normalized_transcript
                ):
                    evidence_matched += 1


# ============================================================
# SAVE CLEAN OUTPUT ONLY
# ============================================================

json_path = (
    output_dir
    / f"{consultation_id}_medgemma_extraction.json"
)

metadata_path = (
    output_dir
    / f"{consultation_id}_medgemma_metadata.json"
)

if json_valid:

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            parsed,
            f,
            indent=2,
            ensure_ascii=False
        )


metadata = {
    "consultation_id": consultation_id,
    "model": "google/medgemma-1.5-4b-it",
    "input_tokens": input_tokens,
    "generated_tokens": generated_tokens,
    "generation_seconds": round(
        generation_seconds,
        2
    ),
    "peak_gpu_gb": round(
        peak_gpu_gb,
        2
    ),
    "json_valid": json_valid,
    "json_error": json_error,
    "evidence_quotes_total": evidence_total,
    "evidence_quotes_matched": evidence_matched,
    "evidence_grounding_percent": (
        round(
            evidence_matched
            / evidence_total
            * 100,
            2
        )
        if evidence_total > 0
        else None
    )
}

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )


# ============================================================
# RESULT
# ============================================================

print("\n" + "=" * 75)
print("MEDGEMMA RETRY RESULT")
print("=" * 75)

print("JSON valid:", json_valid)
print("Generated tokens:", generated_tokens)
print(
    "Generation time:",
    round(generation_seconds, 2),
    "seconds"
)
print(
    "Peak GPU memory:",
    round(peak_gpu_gb, 2),
    "GB"
)

print(
    "Evidence quotes:",
    f"{evidence_matched}/{evidence_total}"
)

if evidence_total:
    print(
        "Evidence grounding:",
        round(
            evidence_matched
            / evidence_total
            * 100,
            2
        ),
        "%"
    )

if json_error:
    print("JSON error:", json_error)

if json_valid:
    print("\nSTRUCTURED JSON")
    print("-" * 75)
    print(
        json.dumps(
            parsed,
            indent=2,
            ensure_ascii=False
        )
    )

print("\nSaved:")
if json_valid:
    print(json_path)
print(metadata_path)


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


MEDGEMMA CLINICAL EXTRACTION RETRY
Input tokens: 2248

MEDGEMMA RETRY RESULT
JSON valid: False
Generated tokens: 3200
Generation time: 95.56 seconds
Peak GPU memory: 8.52 GB
Evidence quotes: 0/0
JSON error: Expecting ',' delimiter: line 205 column 6 (char 6382)

Saved:
/home/jovyan/Case_Study_2_Medical_Consultation_AI/results/nlp/medgemma_1_5_4b_it/compatibility_test/day1_consultation01_medgemma_metadata.json


In [3]:
import json
import re
import time
import torch
from pathlib import Path

print("=" * 75)
print("MEDGEMMA COMPACT CLINICAL EXTRACTION TEST")
print("=" * 75)

consultation_id = "day1_consultation01"

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

transcript_path = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
    / f"{consultation_id}_transcript.txt"
)

output_dir = (
    project
    / "results/nlp/medgemma_1_5_4b_it/compatibility_test"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

transcript = transcript_path.read_text(
    encoding="utf-8"
)

# ============================================================
# COMPACT EXTRACTION PROMPT
# ============================================================

prompt = f"""
Extract the clinically important facts from this medical consultation.

Use ONLY facts explicitly stated in the transcript.

Rules:
- Do not infer or invent information.
- Preserve important negations.
- Preserve numbers, durations, doses and frequencies.
- Combine duplicate or closely related information.
- Extract approximately 15 to 25 important clinical facts.
- Each evidence_quote must be a short quote from the transcript.
- status must be "present" or "absent".
- Return valid JSON only.
- No explanation, reasoning, Markdown or commentary.

Allowed categories:
presenting_complaint
symptom
temporal_detail
medication
allergy
medical_history
family_history
social_history
assessment
plan
safety_netting

Return exactly this structure:

{{
  "consultation_id": "{consultation_id}",
  "clinical_facts": [
    {{
      "category": "",
      "fact": "",
      "status": "present",
      "evidence_quote": ""
    }}
  ]
}}

TRANSCRIPT:

{transcript}
"""

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

input_tokens = inputs["input_ids"].shape[-1]

print("Input tokens:", input_tokens)

# ============================================================
# GENERATION
# ============================================================

torch.cuda.reset_peak_memory_stats()

start = time.time()

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=5000,
        do_sample=False
    )

generation_seconds = time.time() - start

generated_tokens = (
    output.shape[-1]
    - input_tokens
)

full_response = processor.decode(
    output[0][input_tokens:],
    skip_special_tokens=True
).strip()

peak_gpu_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

# Always save raw response for debugging
raw_path = (
    output_dir
    / f"{consultation_id}_medgemma_compact_raw.txt"
)

raw_path.write_text(
    full_response,
    encoding="utf-8"
)

# ============================================================
# ISOLATE FINAL ANSWER
# ============================================================

cleaned = full_response

if "<unused95>" in cleaned:
    cleaned = cleaned.rsplit(
        "<unused95>",
        1
    )[-1]

cleaned = re.sub(
    r"```(?:json)?",
    "",
    cleaned,
    flags=re.IGNORECASE
)

cleaned = cleaned.replace(
    "```",
    ""
).strip()

first = cleaned.find("{")
last = cleaned.rfind("}")

if first >= 0 and last > first:
    json_text = cleaned[first:last + 1]
else:
    json_text = cleaned

# ============================================================
# JSON VALIDATION
# ============================================================

json_valid = False
parsed = None
json_error = None

try:
    parsed = json.loads(json_text)
    json_valid = True
except Exception as e:
    json_error = str(e)

# ============================================================
# EVIDENCE CHECK
# ============================================================

def norm(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

evidence_total = 0
evidence_exact = 0

if json_valid:

    normalized_transcript = norm(
        transcript
    )

    for item in parsed.get(
        "clinical_facts",
        []
    ):
        quote = item.get(
            "evidence_quote",
            ""
        )

        if quote:
            evidence_total += 1

            if norm(quote) in normalized_transcript:
                evidence_exact += 1

# ============================================================
# SAVE
# ============================================================

json_path = (
    output_dir
    / f"{consultation_id}_medgemma_compact_extraction.json"
)

if json_valid:

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            parsed,
            f,
            indent=2,
            ensure_ascii=False
        )

# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 75)
print("MEDGEMMA COMPACT EXTRACTION RESULT")
print("=" * 75)

print("JSON valid:", json_valid)
print("Generated tokens:", generated_tokens)
print(
    "Hit token limit:",
    generated_tokens >= 5000
)

print(
    "Generation time:",
    round(generation_seconds, 2),
    "seconds"
)

print(
    "Peak GPU memory:",
    round(peak_gpu_gb, 2),
    "GB"
)

if json_valid:

    facts = parsed.get(
        "clinical_facts",
        []
    )

    print(
        "Clinical facts extracted:",
        len(facts)
    )

    print(
        "Evidence quotes matched:",
        f"{evidence_exact}/{evidence_total}"
    )

    if evidence_total:
        print(
            "Evidence grounding:",
            round(
                evidence_exact
                / evidence_total
                * 100,
                2
            ),
            "%"
        )

    print("\nSTRUCTURED JSON")
    print("-" * 75)

    print(
        json.dumps(
            parsed,
            indent=2,
            ensure_ascii=False
        )
    )

else:

    print("JSON error:", json_error)

    print("\nLAST 1000 CHARACTERS OF RESPONSE")
    print("-" * 75)
    print(full_response[-1000:])

print("\nRaw response saved:", raw_path)

if json_valid:
    print("JSON saved:", json_path)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


MEDGEMMA COMPACT CLINICAL EXTRACTION TEST
Input tokens: 1974

MEDGEMMA COMPACT EXTRACTION RESULT
JSON valid: True
Generated tokens: 4443
Hit token limit: False
Generation time: 133.18 seconds
Peak GPU memory: 8.46 GB
Clinical facts extracted: 43
Evidence quotes matched: 39/43
Evidence grounding: 90.7 %

STRUCTURED JSON
---------------------------------------------------------------------------
{
  "consultation_id": "day1_consultation01",
  "clinical_facts": [
    {
      "category": "presenting_complaint",
      "fact": "Patient has had diarrhea for the last three days.",
      "status": "present",
      "evidence_quote": "I've just had some diarrhoea for the last three days and it's been affecting me."
    },
    {
      "category": "symptom",
      "fact": "Diarrhea is described as loose and watery stool.",
      "status": "present",
      "evidence_quote": "Do you mean you're going to the toilet more often or are your stools more loose? Yeah, so it's like loose and watery stool..."

In [4]:
import json
import re
from pathlib import Path

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

consultation_id = "day1_consultation01"

json_path = (
    project
    / "results/nlp/medgemma_1_5_4b_it/compatibility_test"
    / f"{consultation_id}_medgemma_compact_extraction.json"
)

transcript_path = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
    / f"{consultation_id}_transcript.txt"
)

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

transcript = transcript_path.read_text(
    encoding="utf-8"
)

def norm(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

normalized_transcript = norm(transcript)

unmatched = []

for i, item in enumerate(
    data.get("clinical_facts", []),
    start=1
):
    quote = item.get("evidence_quote", "")

    if quote and norm(quote) not in normalized_transcript:
        unmatched.append({
            "number": i,
            "category": item.get("category"),
            "fact": item.get("fact"),
            "status": item.get("status"),
            "evidence_quote": quote
        })

print("=" * 75)
print("UNGROUNDED / NON-EXACT EVIDENCE QUOTES")
print("=" * 75)

print("Total facts:", len(data.get("clinical_facts", [])))
print("Unmatched evidence quotes:", len(unmatched))

for item in unmatched:
    print("\n" + "-" * 75)
    print("Fact #:", item["number"])
    print("Category:", item["category"])
    print("Fact:", item["fact"])
    print("Status:", item["status"])
    print("Evidence quote:", item["evidence_quote"])
    

UNGROUNDED / NON-EXACT EVIDENCE QUOTES
Total facts: 43
Unmatched evidence quotes: 4

---------------------------------------------------------------------------
Fact #: 15
Category: temporal_detail
Fact: Symptoms started three days ago.
Status: present
Evidence quote: And you mentioned you've been having some diarrhoea for the last three days...

---------------------------------------------------------------------------
Fact #: 23
Category: medication
Fact: Patient might consider Dirolite (mineral/vitamin supplement).
Status: present
Evidence quote: I would recommend that in the first couple of days. Yeah. If you are having vomiting and diarrhea, I would recommend that in the first couple of days.

---------------------------------------------------------------------------
Fact #: 33
Category: assessment
Fact: Patient likely has gastroenteritis (tummy bug/infection).
Status: present
Evidence quote: I think, just to recap, for the last three days, you've been having loose stool, diarrh

In [5]:
import json
import re
import time
import torch
from pathlib import Path

print("=" * 75)
print("MEDGEMMA FINAL COMPACT EXTRACTION TEST")
print("=" * 75)

consultation_id = "day1_consultation01"

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

transcript_path = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
    / f"{consultation_id}_transcript.txt"
)

output_dir = (
    project
    / "results/nlp/medgemma_1_5_4b_it/final_schema_test"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

transcript = transcript_path.read_text(
    encoding="utf-8"
)

prompt = f"""
Extract only the most clinically important facts from this
doctor-patient consultation transcript.

STRICT RULES:

1. Use ONLY information explicitly present in the transcript.
2. Extract between 15 and 20 facts total.
3. Do not duplicate the same fact in different categories.
4. Prioritize:
   - presenting complaint
   - major symptoms
   - duration/frequency/severity
   - important negations
   - medications and allergies
   - relevant medical history
   - assessment/diagnosis
   - treatment/management
   - follow-up and safety-netting
5. Do not include ordinary social details unless clinically relevant.
6. polarity must be:
   - "present" when the fact is affirmed
   - "absent" when the patient explicitly denies the fact
7. Each evidence_quote must be copied VERBATIM from the transcript.
8. Keep each evidence_quote short, ideally 5-20 words.
9. Do NOT use ellipses (...) inside evidence quotes.
10. Do NOT paraphrase evidence quotes.
11. Preserve important numbers, durations, doses and frequencies exactly.
12. Return valid JSON only.
13. No Markdown, explanation or commentary.

Allowed categories:

presenting_complaint
symptom
temporal_detail
medication
allergy
medical_history
exposure_history
assessment
plan
safety_netting

Return exactly:

{{
  "consultation_id": "{consultation_id}",
  "clinical_facts": [
    {{
      "category": "",
      "fact": "",
      "polarity": "present",
      "importance": "critical",
      "evidence_quote": ""
    }}
  ]
}}

importance must be either:
"critical" or "important"

TRANSCRIPT:

{transcript}
"""

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

input_tokens = inputs["input_ids"].shape[-1]

print("Input tokens:", input_tokens)

torch.cuda.reset_peak_memory_stats()

start = time.time()

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=4000,
        do_sample=False
    )

generation_seconds = time.time() - start

generated_tokens = (
    output.shape[-1] - input_tokens
)

response = processor.decode(
    output[0][input_tokens:],
    skip_special_tokens=True
).strip()

peak_gpu_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

# ------------------------------------------------------------
# Isolate final JSON
# ------------------------------------------------------------

cleaned = response

if "<unused95>" in cleaned:
    cleaned = cleaned.rsplit(
        "<unused95>",
        1
    )[-1].strip()

cleaned = re.sub(
    r"^```(?:json)?\s*",
    "",
    cleaned,
    flags=re.IGNORECASE
)

cleaned = re.sub(
    r"\s*```$",
    "",
    cleaned
).strip()

first = cleaned.find("{")
last = cleaned.rfind("}")

if first >= 0 and last > first:
    json_text = cleaned[first:last + 1]
else:
    json_text = cleaned

# ------------------------------------------------------------
# Parse
# ------------------------------------------------------------

json_valid = False
parsed = None
json_error = None

try:
    parsed = json.loads(json_text)
    json_valid = True
except Exception as e:
    json_error = str(e)

# ------------------------------------------------------------
# Evidence validation
# ------------------------------------------------------------

def norm(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

evidence_total = 0
evidence_matched = 0
unmatched = []

if json_valid:

    normalized_transcript = norm(
        transcript
    )

    facts = parsed.get(
        "clinical_facts",
        []
    )

    for index, item in enumerate(
        facts,
        start=1
    ):

        quote = item.get(
            "evidence_quote",
            ""
        )

        if quote:

            evidence_total += 1

            if norm(quote) in normalized_transcript:
                evidence_matched += 1

            else:
                unmatched.append({
                    "number": index,
                    "fact": item.get("fact"),
                    "quote": quote
                })

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

json_path = (
    output_dir
    / f"{consultation_id}_medgemma_final_extraction.json"
)

metadata_path = (
    output_dir
    / f"{consultation_id}_medgemma_final_metadata.json"
)

if json_valid:

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            parsed,
            f,
            indent=2,
            ensure_ascii=False
        )

metadata = {
    "consultation_id": consultation_id,
    "model": "google/medgemma-1.5-4b-it",
    "input_tokens": input_tokens,
    "generated_tokens": generated_tokens,
    "generation_seconds": round(
        generation_seconds,
        2
    ),
    "peak_gpu_gb": round(
        peak_gpu_gb,
        2
    ),
    "json_valid": json_valid,
    "json_error": json_error,
    "facts_extracted": (
        len(parsed.get("clinical_facts", []))
        if json_valid
        else 0
    ),
    "evidence_total": evidence_total,
    "evidence_matched": evidence_matched,
    "evidence_grounding_percent": (
        round(
            evidence_matched
            / evidence_total
            * 100,
            2
        )
        if evidence_total
        else None
    )
}

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MEDGEMMA FINAL SCHEMA RESULT")
print("=" * 75)

print("JSON valid:", json_valid)
print("Generated tokens:", generated_tokens)
print(
    "Hit token limit:",
    generated_tokens >= 4000
)

if json_valid:

    facts = parsed.get(
        "clinical_facts",
        []
    )

    print(
        "Clinical facts extracted:",
        len(facts)
    )

    print(
        "Evidence grounding:",
        f"{evidence_matched}/{evidence_total}",
        "=",
        round(
            evidence_matched
            / evidence_total
            * 100,
            2
        )
        if evidence_total
        else 0,
        "%"
    )

    print(
        "Ungrounded evidence:",
        len(unmatched)
    )

    if unmatched:

        print("\nUNGROUNDED ITEMS")

        for item in unmatched:
            print("-" * 60)
            print("Fact #:", item["number"])
            print("Fact:", item["fact"])
            print("Quote:", item["quote"])

    print("\nSTRUCTURED JSON")
    print("-" * 75)

    print(
        json.dumps(
            parsed,
            indent=2,
            ensure_ascii=False
        )
    )

else:
    print("JSON error:", json_error)

print("\nGeneration time:", round(generation_seconds, 2), "seconds")
print("Peak GPU:", round(peak_gpu_gb, 2), "GB")

if json_valid:
    print("Saved:", json_path)

print("Saved:", metadata_path)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


MEDGEMMA FINAL COMPACT EXTRACTION TEST
Input tokens: 2133

MEDGEMMA FINAL SCHEMA RESULT
JSON valid: True
Generated tokens: 3709
Hit token limit: False
Clinical facts extracted: 17
Evidence grounding: 15/17 = 88.24 %
Ungrounded evidence: 2

UNGROUNDED ITEMS
------------------------------------------------------------
Fact #: 3
Fact: Patient has frequent bowel movements (6-7 times/day).
Quote: going to the toilet quite often and like some pain in my like lower stomach. Probably like six, seven times a day.
------------------------------------------------------------
Fact #: 4
Fact: Patient has lower abdominal pain, mainly on the left side.
Quote: pain in my like lower stomach. So it's like in my lower abdomen, so like, yeah, just to one side. One side, and what side is that? On the left side.

STRUCTURED JSON
---------------------------------------------------------------------------
{
  "consultation_id": "day1_consultation01",
  "clinical_facts": [
    {
      "category": "presenting_c

In [6]:
import json
import re
import time
import torch
from pathlib import Path

print("=" * 75)
print("MEDGEMMA BALANCED CLINICAL EXTRACTION TEST")
print("=" * 75)

consultation_id = "day1_consultation01"

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

transcript_path = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
    / f"{consultation_id}_transcript.txt"
)

output_dir = (
    project
    / "results/nlp/medgemma_1_5_4b_it/balanced_schema_test"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

transcript = transcript_path.read_text(
    encoding="utf-8"
)

prompt = f"""
Extract a concise set of clinically important facts from this
doctor-patient consultation.

Use ONLY information explicitly stated in the transcript.

IMPORTANT: Read the ENTIRE consultation, including the doctor's
assessment, treatment plan and safety-netting near the end.

Return approximately 15 to 20 facts total.

PRIORITY BALANCE:
- 1 presenting complaint if present
- important symptoms, duration, frequency and severity
- clinically important NEGATIVE findings
- relevant medications, allergies and medical history
- relevant exposure history
- assessment or working diagnosis if stated
- treatment or management plan if stated
- follow-up or safety-netting if stated

Do not fill all available facts with symptoms from the beginning
of the consultation. Reserve space for assessment and plan.

CATEGORY DEFINITIONS:

presenting_complaint:
main reason for consultation

symptom:
positive or negative symptom/finding

temporal_detail:
important duration, onset, frequency or timing

medication:
current medication or specifically recommended medication

allergy:
drug/substance allergy ONLY

medical_history:
past or current medical condition ONLY

exposure_history:
relevant food, travel, infectious or environmental exposure

social_history:
smoking, alcohol, occupation or living situation ONLY when
clinically relevant

assessment:
doctor's diagnosis, impression or differential diagnosis

plan:
treatment, medication advice, investigation or management

safety_netting:
follow-up instructions, return precautions or escalation advice

POLARITY RULE:
- "present" means the clinical fact is affirmed.
- "absent" means the patient explicitly denies the condition/finding.

Examples:
"No blood in vomit" -> polarity = "absent"
"Patient does not smoke" -> polarity = "absent"
"Patient has asthma" -> polarity = "present"

Do NOT classify:
- smoking/alcohol as allergy
- occupation or family living situation as medical history
- recommendations as current medication
- doctor questions as confirmed patient facts

EVIDENCE RULES:
- Every evidence_quote must be copied verbatim from ONE continuous
  span of the transcript.
- Maximum approximately 20 words.
- Do not join separate transcript fragments.
- Do not use ellipses.
- Do not paraphrase the evidence quote.

Return valid JSON only.
No reasoning, explanation, Markdown or commentary.

Allowed categories:
presenting_complaint
symptom
temporal_detail
medication
allergy
medical_history
exposure_history
social_history
assessment
plan
safety_netting

Return:

{{
  "consultation_id": "{consultation_id}",
  "clinical_facts": [
    {{
      "category": "",
      "fact": "",
      "polarity": "present",
      "importance": "critical",
      "evidence_quote": ""
    }}
  ]
}}

importance must be:
"critical" or "important"

TRANSCRIPT:

{transcript}
"""

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

input_tokens = inputs["input_ids"].shape[-1]

print("Input tokens:", input_tokens)

torch.cuda.reset_peak_memory_stats()

start = time.time()

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=4000,
        do_sample=False
    )

generation_seconds = time.time() - start

generated_tokens = (
    output.shape[-1] - input_tokens
)

response = processor.decode(
    output[0][input_tokens:],
    skip_special_tokens=True
).strip()

peak_gpu_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

# ============================================================
# CLEAN FINAL RESPONSE
# ============================================================

cleaned = response

if "<unused95>" in cleaned:
    cleaned = cleaned.rsplit(
        "<unused95>",
        1
    )[-1].strip()

cleaned = re.sub(
    r"^```(?:json)?\s*",
    "",
    cleaned,
    flags=re.IGNORECASE
)

cleaned = re.sub(
    r"\s*```$",
    "",
    cleaned
).strip()

first = cleaned.find("{")
last = cleaned.rfind("}")

if first >= 0 and last > first:
    json_text = cleaned[first:last + 1]
else:
    json_text = cleaned

# ============================================================
# PARSE
# ============================================================

json_valid = False
parsed = None
json_error = None

try:
    parsed = json.loads(json_text)
    json_valid = True
except Exception as e:
    json_error = str(e)

# ============================================================
# VALIDATION
# ============================================================

def norm(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

evidence_total = 0
evidence_matched = 0
unmatched = []

category_counts = {}

if json_valid:

    normalized_transcript = norm(transcript)

    facts = parsed.get(
        "clinical_facts",
        []
    )

    for i, item in enumerate(
        facts,
        start=1
    ):

        category = item.get(
            "category",
            ""
        )

        category_counts[category] = (
            category_counts.get(
                category,
                0
            ) + 1
        )

        quote = item.get(
            "evidence_quote",
            ""
        )

        if quote:

            evidence_total += 1

            if norm(quote) in normalized_transcript:
                evidence_matched += 1
            else:
                unmatched.append({
                    "number": i,
                    "category": category,
                    "fact": item.get("fact"),
                    "polarity": item.get("polarity"),
                    "quote": quote
                })

# ============================================================
# SAVE
# ============================================================

json_path = (
    output_dir
    / f"{consultation_id}_medgemma_balanced_extraction.json"
)

metadata_path = (
    output_dir
    / f"{consultation_id}_medgemma_balanced_metadata.json"
)

if json_valid:

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            parsed,
            f,
            indent=2,
            ensure_ascii=False
        )

metadata = {
    "consultation_id": consultation_id,
    "model": "google/medgemma-1.5-4b-it",
    "json_valid": json_valid,
    "json_error": json_error,
    "input_tokens": input_tokens,
    "generated_tokens": generated_tokens,
    "generation_seconds": round(
        generation_seconds,
        2
    ),
    "peak_gpu_gb": round(
        peak_gpu_gb,
        2
    ),
    "facts_extracted": (
        len(parsed.get("clinical_facts", []))
        if json_valid
        else 0
    ),
    "evidence_grounding_percent": (
        round(
            evidence_matched
            / evidence_total
            * 100,
            2
        )
        if evidence_total
        else None
    ),
    "category_counts": category_counts
}

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

# ============================================================
# RESULT
# ============================================================

print("\n" + "=" * 75)
print("MEDGEMMA BALANCED SCHEMA RESULT")
print("=" * 75)

print("JSON valid:", json_valid)
print("Generated tokens:", generated_tokens)
print(
    "Hit token limit:",
    generated_tokens >= 4000
)

if json_valid:

    facts = parsed.get(
        "clinical_facts",
        []
    )

    print(
        "Clinical facts extracted:",
        len(facts)
    )

    print(
        "Evidence grounding:",
        f"{evidence_matched}/{evidence_total}",
        "=",
        round(
            evidence_matched
            / evidence_total
            * 100,
            2
        )
        if evidence_total
        else 0,
        "%"
    )

    print("\nCATEGORY COUNTS")
    print("-" * 40)

    for category, count in sorted(
        category_counts.items()
    ):
        print(
            f"{category}: {count}"
        )

    print("\nSTRUCTURED JSON")
    print("-" * 75)

    print(
        json.dumps(
            parsed,
            indent=2,
            ensure_ascii=False
        )
    )

    if unmatched:
        print("\nUNGROUNDED ITEMS")
        print("-" * 75)

        for item in unmatched:
            print(item)

else:
    print(
        "JSON error:",
        json_error
    )

print(
    "\nGeneration time:",
    round(generation_seconds, 2),
    "seconds"
)

print(
    "Peak GPU:",
    round(peak_gpu_gb, 2),
    "GB"
)

if json_valid:
    print("Saved:", json_path)

print("Saved:", metadata_path)


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


MEDGEMMA BALANCED CLINICAL EXTRACTION TEST
Input tokens: 2378

MEDGEMMA BALANCED SCHEMA RESULT
JSON valid: False
Generated tokens: 4000
Hit token limit: True
JSON error: Expecting ',' delimiter: line 45 column 6 (char 1874)

Generation time: 119.9 seconds
Peak GPU: 8.55 GB
Saved: /home/jovyan/Case_Study_2_Medical_Consultation_AI/results/nlp/medgemma_1_5_4b_it/balanced_schema_test/day1_consultation01_medgemma_balanced_metadata.json
